In [4]:
import numpy as np
import pandas as pd
import importlib
from pathlib import Path

import data_pipeline
importlib.reload(data_pipeline)
from data_pipeline import DataPipeline, load_data, train_test_split

## Task A1: EDA khảo sát và chẩn đoán

In [5]:
data_path = Path("data/melb_data.csv")
df = load_data(data_path)
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

ghi chú: Schema gồm numeric và categorical; `Price` là target, các bước sửa dữ liệu chưa được áp dụng ở block này.


In [7]:
df.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


ghi chú: Các cột numeric có đơn vị và biên độ khác nhau; pipeline cần chuẩn hóa Z-score sau khi encode để đưa feature về cùng hệ đo.


In [8]:
missing_values = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
missing_values

,missing_count,missing_pct
BuildingArea,6450,47.50
YearBuilt,5375,39.58
CouncilArea,1369,10.08
Car,62,0.46


ghi chú: Missing tập trung ở `BuildingArea`, `YearBuilt`, `CouncilArea`, `Car`; numeric missing cần xử lý sau split bằng fitted imputer, `CouncilArea` không đưa vào model matrix theo contract hiện tại.


In [9]:
outlier_columns = ["Price", "Landsize", "BuildingArea", "YearBuilt"]
outlier_detection = pd.DataFrame(
    [
        {"check": "Price > 99th percentile", "rows": int((df["Price"] > df["Price"].quantile(0.99)).sum())},
        {"check": "Landsize == 0", "rows": int((df["Landsize"] == 0).sum())},
        {"check": "Landsize > 10000", "rows": int((df["Landsize"] > 10000).sum())},
        {"check": "BuildingArea <= 0", "rows": int((df["BuildingArea"] <= 0).sum())},
        {"check": "BuildingArea > 1000", "rows": int((df["BuildingArea"] > 1000).sum())},
        {"check": "YearBuilt < 1800", "rows": int((df["YearBuilt"] < 1800).sum())},
    ]
)
outlier_detection

,check,rows
0,Price > 99th percentile,136
1,Landsize == 0,1939
2,Landsize > 10000,26
3,BuildingArea <= 0,17
4,BuildingArea > 1000,8
5,YearBuilt < 1800,1


ghi chú: `BuildingArea <= 0`, `YearBuilt < 1800`, `Landsize <= 0` không phù hợp về mặt giá trị đo; pipeline chuyển các giá trị này thành missing trước khi impute hoặc tính ratio.


ghi chú: A1 chỉ khảo sát dữ liệu thô; chưa sửa missing value, invalid value, scale hoặc encode.


## Task A2: Chia tách dữ liệu thô

In [10]:
X_train_raw, X_test_raw = train_test_split(df, test_size=0.3, random_state=42)

split_summary = pd.DataFrame(
    {
        "split": ["X_train_raw", "X_test_raw"],
        "rows": [len(X_train_raw), len(X_test_raw)],
        "pct": [round(len(X_train_raw) / len(df) * 100, 2), round(len(X_test_raw) / len(df) * 100, 2)],
    }
)
split_summary

,split,rows,pct
0,X_train_raw,9506,70.0
1,X_test_raw,4074,30.0


ghi chú: Split được thực hiện trên DataFrame thô; mọi trạng thái fit của imputer, encoder và scaler chỉ học từ `X_train_raw`.


## EDA trên train split


In [11]:
target_distribution_check = pd.DataFrame(
    {
        "X_train_raw": X_train_raw["Price"].describe(percentiles=[0.05, 0.5, 0.95]),
        "X_test_raw": X_test_raw["Price"].describe(percentiles=[0.05, 0.5, 0.95]),
    }
).round(2)
target_distribution_check


,X_train_raw,X_test_raw
count,9506.00,4074.00
mean,1076128.42,1074647.30
std,637183.57,644324.13
min,131000.00,85000.00
5%,407125.00,400000.00
50%,905000.00,900250.00
95%,2300000.00,2261750.00
max,8000000.00,9000000.00


ghi chú: `Price` không thiếu giá trị; `y_train` và `y_test` được tách trực tiếp từ split thô, không biến đổi target trong pipeline.


In [12]:
train_missing_summary = pd.DataFrame(
    {
        "missing_count": X_train_raw.isna().sum(),
        "missing_pct": X_train_raw.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
train_missing_summary


,missing_count,missing_pct
BuildingArea,4557,47.94
YearBuilt,3815,40.13
CouncilArea,953,10.03
Car,42,0.44


ghi chú: Missing numeric được xử lý bằng `KNNImputer` fit trên `X_train_raw`; các cột thiếu nhiều như `BuildingArea` và `YearBuilt` có thêm missing flag trước imputation.


In [13]:
train_invalid_summary = pd.DataFrame(
    [
        {"check": "Price <= 0", "rows": int((X_train_raw["Price"] <= 0).sum())},
        {"check": "Landsize <= 0", "rows": int((X_train_raw["Landsize"] <= 0).sum())},
        {"check": "Landsize > 10000", "rows": int((X_train_raw["Landsize"] > 10000).sum())},
        {"check": "BuildingArea <= 0", "rows": int((X_train_raw["BuildingArea"] <= 0).sum())},
        {"check": "BuildingArea > 1000", "rows": int((X_train_raw["BuildingArea"] > 1000).sum())},
        {"check": "YearBuilt < 1800", "rows": int((X_train_raw["YearBuilt"] < 1800).sum())},
    ]
)
train_invalid_summary


,check,rows
0,Price <= 0,0
1,Landsize <= 0,1353
2,Landsize > 10000,21
3,BuildingArea <= 0,13
4,BuildingArea > 1000,7
5,YearBuilt < 1800,1


ghi chú: Outlier giá trị lớn chỉ dùng để nhận diện phân phối lệch; pipeline không drop row outlier, chỉ repair các giá trị không hợp lệ đã xác định.


In [14]:
numeric_columns = [
    "Price",
    "Rooms",
    "Bedroom2",
    "Bathroom",
    "Car",
    "Distance",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Propertycount",
]

numeric_profile = X_train_raw[numeric_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_profile["skew"] = X_train_raw[numeric_columns].skew(numeric_only=True)
numeric_profile.round(2)


,count,mean,std,min,1%,5%,50%,95%,99%,max,skew
Price,9506.0,1076128.42,637183.57,131000.0,300050.00,407125.0,905000.0,2300000.0,3359500.00,8000000.0,2.19
Rooms,9506.0,2.94,0.95,1.0,1.00,2.0,3.0,4.0,5.00,10.0,0.35
Bedroom2,9506.0,2.93,0.97,0.0,1.00,1.0,3.0,4.0,5.00,20.0,0.92
Bathroom,9506.0,1.53,0.69,0.0,1.00,1.0,1.0,3.0,3.95,8.0,1.32
Car,9464.0,1.60,0.95,0.0,0.00,0.0,2.0,3.0,4.00,10.0,1.34
Distance,9506.0,10.12,5.90,0.0,1.60,2.6,9.2,20.6,32.27,48.1,1.67
Landsize,9506.0,541.64,1642.07,0.0,0.00,0.0,446.0,1002.5,2976.85,76000.0,28.97
BuildingArea,4949.0,154.42,646.59,0.0,6.92,52.4,126.0,291.0,446.60,44515.0,65.56
YearBuilt,5691.0,1964.41,37.74,1196.0,1880.00,1900.0,1970.0,2012.0,2015.00,2018.0,-1.94
Propertycount,9506.0,7490.50,4385.61,249.0,962.00,2211.0,6567.0,14949.0,21650.00,21650.0,1.08


ghi chú: `Landsize`, `BuildingArea`, `Distance` có scale lệch mạnh so với các biến đếm; bước scaling phải chạy sau imputation, feature engineering và one-hot để chuẩn hóa toàn bộ ma trận cuối.


In [15]:
relationship_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance", "BuildingArea", "YearBuilt"]
relationship_summary = X_train_raw[relationship_columns].corr(numeric_only=True)[["Price", "Rooms"]].round(3)
relationship_summary


,Price,Rooms
Price,1.000,0.499
Rooms,0.499,1.000
Bedroom2,0.480,0.941
Bathroom,0.481,0.593
Car,0.238,0.402
Distance,-0.164,0.298
BuildingArea,0.079,0.112
YearBuilt,-0.331,-0.075


ghi chú: `Bedroom2` chồng lấp với `Rooms`; cột này được đưa vào `drop_columns`, còn `Rooms`, `Bathroom` và `OtherRooms` giữ lại tín hiệu quy mô và bố cục.


In [16]:
categorical_columns = ["Type", "Regionname", "Method", "CouncilArea"]
categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [X_train_raw[column].nunique(dropna=True) for column in categorical_columns],
        "missing_pct": [round(X_train_raw[column].isna().mean() * 100, 2) for column in categorical_columns],
        "top_value": [X_train_raw[column].mode(dropna=True).iloc[0] for column in categorical_columns],
    }
)
categorical_summary


,column,unique_values,missing_pct,top_value
0,Type,3,0.00,h
1,Regionname,8,0.00,Southern Metropolitan
2,Method,5,0.00,S
3,CouncilArea,33,10.03,Moreland


ghi chú: `Type` và `Regionname` có cardinality thấp, phù hợp one-hot; `Method` và `CouncilArea` bị drop để giữ đúng phạm vi encoding của task A4.


In [17]:
feature_preview = X_train_raw[["Rooms", "Bathroom", "BuildingArea", "Landsize", "YearBuilt"]].copy()
feature_preview["Age"] = 2026 - feature_preview["YearBuilt"]
feature_preview["OtherRooms"] = (feature_preview["Rooms"] - feature_preview["Bathroom"]).clip(lower=0)
feature_preview["BuildingArea_per_Room"] = feature_preview["BuildingArea"] / feature_preview["Rooms"].replace(0, np.nan)
feature_preview["BuildingCoverage"] = feature_preview["BuildingArea"] / feature_preview["Landsize"].replace(0, np.nan)
feature_preview[["Age", "OtherRooms", "BuildingArea_per_Room", "BuildingCoverage"]].describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
Age,5691.0,61.59,37.74,8.0,27.00,56.00,86.00,830.0
OtherRooms,9506.0,1.41,0.77,0.0,1.00,1.00,2.00,7.0
BuildingArea_per_Room,4949.0,50.03,132.68,0.0,37.33,44.00,53.00,8903.0
BuildingCoverage,4214.0,0.44,1.21,0.0,0.22,0.34,0.54,62.0


ghi chú: Feature tạo thêm chỉ dùng phép biến đổi đơn giản từ cột có sẵn; các ratio phát sinh NaN hoặc inf sẽ được xử lý bằng fallback numeric đã fit từ train.


## Tasks A3-A6: Thực thi pipeline

In [18]:
pipeline = DataPipeline(drop_columns=["Bedroom2"])
X_train, y_train = pipeline.fit_transform(X_train_raw)
X_test, y_test = pipeline.transform(X_test_raw)

metadata = {
    "feature_names": pipeline.feature_names,
    "target_name": pipeline.target_name,
    "pipeline": pipeline,
    "imputation_strategy": "knn_imputer",
    "scaling_method": "standardize",
    "encoded_categorical_columns": pipeline.categorical_columns,
    "encoded_columns": pipeline.encoded_columns,
    "drop_columns": pipeline.drop_columns,
    "engineered_features": [
        "Age",
        "OtherRooms",
        "BuildingArea_per_Room",
        "BuildingCoverage",
        "BuildingArea_missing",
        "YearBuilt_missing",
        "Landsize_zero_or_missing",
    ],
}

pd.DataFrame(
    {
        "artifact": ["X_train", "X_test", "y_train", "y_test", "features"],
        "shape": [X_train.shape, X_test.shape, y_train.shape, y_test.shape, len(metadata["feature_names"])],
    }
)

,artifact,shape
0,X_train,"(9506, 28)"
1,X_test,"(4074, 28)"
2,y_train,"(9506,)"
3,y_test,"(4074,)"
4,features,28


## Feature contract validation

In [19]:
expected_engineered_features = metadata["engineered_features"]
X_train_df = pd.DataFrame(X_train, columns=metadata["feature_names"])
X_test_df = pd.DataFrame(X_test, columns=metadata["feature_names"])

feature_contract_check = pd.DataFrame(
    {
        "feature": expected_engineered_features,
        "exists_in_output": [feature in metadata["feature_names"] for feature in expected_engineered_features],
        "train_has_no_nan": [
            bool(not X_train_df[feature].isna().any()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_has_no_nan": [
            bool(not X_test_df[feature].isna().any()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
        "train_is_finite": [
            bool(np.isfinite(X_train_df[feature]).all()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_is_finite": [
            bool(np.isfinite(X_test_df[feature]).all()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
    }
)

failed_feature_contract = feature_contract_check.loc[
    ~feature_contract_check.drop(columns=["feature"]).all(axis=1)
]
assert failed_feature_contract.empty, failed_feature_contract.to_string(index=False)
feature_contract_check

,feature,exists_in_output,train_has_no_nan,test_has_no_nan,train_is_finite,test_is_finite
0,Age,True,True,True,True,True
1,OtherRooms,True,True,True,True,True
2,BuildingArea_per_Room,True,True,True,True,True
3,BuildingCoverage,True,True,True,True,True
4,BuildingArea_missing,True,True,True,True,True
5,YearBuilt_missing,True,True,True,True,True
6,Landsize_zero_or_missing,True,True,True,True,True


## Handoff validation

In [20]:
handoff_checks = pd.DataFrame(
    [
        {"check": "X_train has no NaN", "passed": bool(not np.isnan(X_train).any())},
        {"check": "X_test has no NaN", "passed": bool(not np.isnan(X_test).any())},
        {"check": "X_train is finite", "passed": bool(np.isfinite(X_train).all())},
        {"check": "X_test is finite", "passed": bool(np.isfinite(X_test).all())},
        {"check": "same feature count", "passed": bool(X_train.shape[1] == X_test.shape[1])},
        {"check": "metadata feature count", "passed": bool(len(metadata["feature_names"]) == X_train.shape[1])},
        {"check": "y_train has no NaN", "passed": bool(not np.isnan(y_train).any())},
        {"check": "y_test has no NaN", "passed": bool(not np.isnan(y_test).any())},
        {"check": "y_train is finite", "passed": bool(np.isfinite(y_train).all())},
        {"check": "y_test is finite", "passed": bool(np.isfinite(y_test).all())},
        {"check": "X_train/y_train aligned", "passed": bool(X_train.shape[0] == len(y_train))},
        {"check": "X_test/y_test aligned", "passed": bool(X_test.shape[0] == len(y_test))},
    ]
)

assert handoff_checks["passed"].all()
handoff_checks

,check,passed
0,X_train has no NaN,True
1,X_test has no NaN,True
2,X_train is finite,True
3,X_test is finite,True
4,same feature count,True
5,metadata feature count,True
6,y_train has no NaN,True
7,y_test has no NaN,True
8,y_train is finite,True
9,y_test is finite,True


In [21]:
pd.DataFrame({"feature_name": metadata["feature_names"]})

,feature_name
0,Rooms
1,Distance
2,Bathroom
3,Car
4,Landsize
5,BuildingArea
6,YearBuilt
7,Lattitude
8,Longtitude
9,Propertycount
